# Example 03: Non-streaming vs Streaming Latency（延迟对比）

测量首 token 时延（TTFT）与总耗时，量化流式的优势

**Demonstrates:**
- Measuring Time-to-First-Token (TTFT) with streaming
- Measuring total round-trip latency for both modes
- Why streaming is preferred for interactive use cases

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server first (see quickstart.md)
```

In [ ]:
import os
import time
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
    timeout=120,
)

MODEL = os.environ.get("HY3_MODEL", "hy3")
PROMPT = "Explain the difference between TCP and UDP in 3 bullet points."

## Non-streaming（非流式）

等待模型生成完整回复后一次性返回，TTFT 无法独立测量。

In [ ]:
t_start = time.perf_counter()

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    temperature=0.9,
    top_p=1.0,
    stream=False,
    extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
)

total_s = time.perf_counter() - t_start
content = response.choices[0].message.content
completion_tokens = response.usage.completion_tokens

print(f"Content:\n{content}")
print(f"\nMetrics:")
print(f"  Time-to-first-token : N/A (non-streaming)")
print(f"  Total latency        : {total_s:.3f}s")
print(f"  Completion tokens    : {completion_tokens}")
print(f"  Throughput           : {completion_tokens / total_s:.1f} tokens/s")

**Sample output:**
```
Content:
- **TCP** is connection-oriented, guarantees delivery and ordering via handshake and ACKs.
- **UDP** is connectionless, sends packets without confirmation — faster but unreliable.
- **Use TCP** for file transfers and web; **use UDP** for video streaming and gaming.

Metrics:
  Time-to-first-token : N/A (non-streaming)
  Total latency        : 4.821s
  Completion tokens    : 68
  Throughput           : 14.1 tokens/s
```

## Streaming（流式）

第一个 token 到达时立即记录 TTFT，总延迟与非流式相近，但用户体验差异显著。

In [ ]:
t_start = time.perf_counter()
t_first_token = None
full_content = ""
completion_tokens = 0

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    temperature=0.9,
    top_p=1.0,
    stream=True,
    stream_options={"include_usage": True},
    extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
)

for chunk in stream:
    if chunk.usage is not None:
        completion_tokens = chunk.usage.completion_tokens
        continue
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta.content:
        if t_first_token is None:
            t_first_token = time.perf_counter()  # record TTFT
        full_content += delta.content

total_s = time.perf_counter() - t_start
ttft_s = (t_first_token - t_start) if t_first_token else None

print(f"Content:\n{full_content}")
print(f"\nMetrics:")
ttft_str = f"{ttft_s:.3f}s" if ttft_s else "N/A"
print(f"  Time-to-first-token : {ttft_str}")
print(f"  Total latency        : {total_s:.3f}s")
print(f"  Completion tokens    : {completion_tokens}")
print(f"  Throughput           : {completion_tokens / total_s:.1f} tokens/s")

**Sample output:**
```
Content:
- **TCP** is connection-oriented, guarantees delivery and ordering via handshake and ACKs.
- **UDP** is connectionless, sends packets without confirmation — faster but unreliable.
- **Use TCP** for file transfers and web; **use UDP** for video streaming and gaming.

Metrics:
  Time-to-first-token : 0.347s
  Total latency        : 4.893s
  Completion tokens    : 68
  Throughput           : 13.9 tokens/s
```

## Summary（总结）

| Mode | TTFT | Total latency | Best for |
|---|---|---|---|
| Non-streaming | N/A | ~5s | Batch processing, offline pipelines |
| **Streaming** | **~0.35s** | ~5s | **Interactive UIs, chatbots** |

> **Key insight**: 流式的总耗时与非流式几乎相同，但 TTFT 降低 93%——用户看到第一个字的等待时间从 5s 降到 0.35s，体验差异极大。